#Sampling Kinematic Variables

In [ ]:
import photoproductionmodel as pm

In [ ]:
def mcmc_sample_EX_cosX(N,s_tot,params,mX,xq):
  '''
  N: int number of samples
  s_tot: float experimental parameter
  params: np.array([floats]) photoproduction model parameters
  mX: float mass of X boson
  xq: np.array([floats]) quark gauge charges
  '''

  d = 2
  x_samples = np.zeros((N, d))
  E_beam = (s_tot - mp**2) / (2 * mp)
  x_samples[0] = np.array([(E_beam - mX)/2,0.99]) #make median energy and angle the initial sample
  counter = 0

  for i in range(N - 1):
    EX_old,costheta_old = x_samples[i]

    ## proposal ##


    EX_new = np.random.uniform(mX,E_beam)
    costheta_new = np.random.uniform(-1,1)
    qX = np.sqrt(EX_new**2 - mX**2)
    nu = (mp * EX_new - 0.5 * mX**2) / (mp - EX_new + qX * costheta_new)

    while any([EX_new < mX, (nu <= EX_new), (nu >= E_beam) , (nu <= 0)]):
      EX_new = np.random.uniform(mX,E_beam)
      costheta_new = np.random.uniform(-1,1)
      qX = np.sqrt(EX_new**2 - mX**2)
      nu = (mp * EX_new - 0.5 * mX**2) / (mp - EX_new + qX * costheta_new)

    ## accept/reject for log ##
    A = P(s_tot,EX_new,costheta_new,params,mX,xq) / P(s_tot,EX_old,costheta_old,params,mX,xq)

    if A > 1:
      x_samples[i+1] = np.array([EX_new,costheta_new])
      counter +=1

    elif A > np.random.uniform(0,1):
      x_samples[i+1] = np.array([EX_new,costheta_new])
      counter +=1
    else:
      x_samples[i+1] = np.array([EX_old,costheta_old])

  return x_samples , counter/N

def P(s_tot,EX,cosX,params,mX,xq):
  return pm.dsigX_dEX_dcosX(s_tot,EX,cosX,params,mX,xq)

In [ ]:
N = 10000 # number of samples at each mass
s_tot = 70 # experimental parameter
params = params # converged parameters set in initialization
xq = gauge_couplings["Chargephobic"][0:3] # quark charges

resolution = 10 # number of divisions of mass range (masses we are sampling at)
mX_range = np.linspace(0,2,resolution) # mass range object iterated on to call sampler, it's length is equal to resolution, its contains the actual mass values

out = [mcmc_sample_EX_cosX(N,s_tot,params,mX,xq) for mX in mX_range]

samples = np.zeros(N,dtype=object) # array containing arrays of [EX,costh] samples corresponding to mX_range. i.e. samples[0] returns an array of [EX,costh] samples for the first mass value in mX_range
for i in range(0,len(mX_range)):
  samples[i] = out[i][0]

acceptances = np.zeros(len(mX_range))
for i in range(0,len(mX_range)):
  acceptances[i] = out[i][1]


In [ ]:
print('num of samples: ' + str(10000*acceptances))